In [ ]:
# ONE-CELL GPU EXTENSION: matched compound-corruption evaluation
# Edit only DATA_ROOT and CHECKPOINT_PATH below, then run this entire cell.
# No training is performed. Use the held-out TEST split only.

import os, io, json, time, random, zipfile, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageEnhance
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

# -------------------- USER SETTINGS --------------------
DATA_ROOT = "/content/drive/MyDrive/imagenette/test"  # class-folder TEST split only
CHECKPOINT_PATH = "/content/drive/MyDrive/resnet18_checkpoint.pth"  # existing trained ResNet-18
OUTPUT_DIR = "/content/compound_robustness_results"
MAX_IMAGES = 0       # 0 = all images; use 500 for a quick first run
BATCH_SIZE = 48
NUM_WORKERS = 2
SEED = 42
# -------------------------------------------------------

start_time = time.time()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU unavailable; this run will use CPU.")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

data_root = Path(DATA_ROOT)
ckpt_path = Path(CHECKPOINT_PATH)
out_dir = Path(OUTPUT_DIR); out_dir.mkdir(parents=True, exist_ok=True)
if not data_root.exists(): raise FileNotFoundError(f"TEST dataset folder not found: {data_root}")
if not ckpt_path.exists(): raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

# ImageNet preprocessing recorded for the original ResNet18 evaluation:
# Resize(256), CenterCrop(224), ImageNet normalization.
preprocess = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# Discover class folders and images. Keep this exact source set for all conditions.
extensions = {".jpg",".jpeg",".png",".bmp",".webp"}
classes = sorted([p.name for p in data_root.iterdir() if p.is_dir()])
if not classes: raise ValueError("Expected ImageFolder structure: TEST_ROOT/class_name/image.jpg")
class_to_idx = {c:i for i,c in enumerate(classes)}
records=[]
for cls in classes:
    for fp in sorted((data_root/cls).rglob("*")):
        if fp.is_file() and fp.suffix.lower() in extensions:
            records.append((fp, class_to_idx[cls], cls))
if MAX_IMAGES and len(records)>MAX_IMAGES:
    # deterministic, class-balanced subset
    rng=np.random.default_rng(SEED)
    chosen=[]
    for ci in range(len(classes)):
        ix=[i for i,r in enumerate(records) if r[1]==ci]
        if ix:
            chosen.extend(rng.choice(ix, size=min(len(ix), max(1, MAX_IMAGES//len(classes))), replace=False).tolist())
    records=[records[i] for i in sorted(set(chosen))][:MAX_IMAGES]
if not records: raise ValueError("No supported image files found.")
print(f"Images: {len(records)} | classes: {classes}")

# Load a ResNet-18 checkpoint. Supports raw state_dict or common wrapped formats.
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(classes))
obj = torch.load(str(ckpt_path), map_location="cpu")
if isinstance(obj, nn.Module):
    model = obj
else:
    state = obj
    if isinstance(obj, dict):
        for key in ["state_dict","model_state_dict","model","weights"]:
            if key in obj and isinstance(obj[key], dict):
                state=obj[key]; break
    if not isinstance(state, dict): raise ValueError("Unrecognized checkpoint format.")
    state = {k.replace("module.","",1):v for k,v in state.items()}
    # Some checkpoints store classifier under fc and others may contain extra metadata.
    missing, unexpected = model.load_state_dict(state, strict=False)
    print("Checkpoint loaded. Missing keys:", missing[:8], "| Unexpected keys:", unexpected[:8])
    if len(missing)>4:
        raise RuntimeError("Checkpoint did not load into this ResNet18 head. Check architecture/class count and checkpoint format.")
model = model.to(device).eval()

# Deterministic paired corruptions. Noise is applied in uint8 RGB space.
def jpeg_roundtrip(im, quality=50):
    b=io.BytesIO(); im.convert("RGB").save(b, format="JPEG", quality=quality); b.seek(0)
    with Image.open(b) as x: return x.convert("RGB").copy()

def add_noise(im, sigma=12.0, seed=0):
    a=np.asarray(im.convert("RGB"), dtype=np.float32)
    rng=np.random.default_rng(seed)
    a=np.clip(a+rng.normal(0,sigma,a.shape),0,255).astype(np.uint8)
    return Image.fromarray(a, "RGB")

def make_condition(im, condition, seed):
    im=im.convert("RGB")
    if condition=="clean": return im
    if condition=="jpeg_q50": return jpeg_roundtrip(im,50)
    if condition=="gaussian_noise": return add_noise(im,12.0,seed)
    if condition=="gaussian_blur": return im.filter(ImageFilter.GaussianBlur(radius=1.2))
    if condition=="brightness_up": return ImageEnhance.Brightness(im).enhance(1.25)
    if condition=="noise_plus_blur":
        return add_noise(im.filter(ImageFilter.GaussianBlur(radius=1.2)),12.0,seed)
    if condition=="jpeg_q50_plus_noise":
        return add_noise(jpeg_roundtrip(im,50),12.0,seed)
    if condition=="jpeg_q50_plus_blur":
        return jpeg_roundtrip(im.filter(ImageFilter.GaussianBlur(radius=1.2)),50)
    raise ValueError(condition)

conditions=["clean","jpeg_q50","gaussian_noise","gaussian_blur","brightness_up",
            "noise_plus_blur","jpeg_q50_plus_noise","jpeg_q50_plus_blur"]

class PairedConditionDataset(Dataset):
    def __init__(self, rows, condition):
        self.rows=rows; self.condition=condition
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        path,label,cls=self.rows[i]
        with Image.open(path) as im: im=im.convert("RGB")
        seed=int.from_bytes(__import__("hashlib").sha256((str(path)+str(SEED)).encode()).digest()[:4],"little")
        im=make_condition(im,self.condition,seed)
        return preprocess(im), label, str(path), cls

all_rows=[]
for condition in conditions:
    ds=PairedConditionDataset(records,condition)
    loader=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
                      pin_memory=(device.type=="cuda"))
    for xb,yb,paths,cls_names in loader:
        xb=xb.to(device,non_blocking=True)
        with torch.inference_mode():
            logits=model(xb)
            probs=torch.softmax(logits,dim=1)
            conf,pred=probs.max(dim=1)
        preds=pred.cpu().numpy(); confs=conf.cpu().numpy(); ys=yb.numpy()
        for path,y,pred_i,conf_i,cl in zip(paths,ys,preds,confs,cls_names):
            all_rows.append({"source_id":path,"label":int(y),"class_name":cl,
                             "condition":condition,"prediction":int(pred_i),
                             "confidence":float(conf_i),"correct":int(int(y)==int(pred_i))})
    print(f"Completed {condition}: {sum(r['condition']==condition for r in all_rows)} images")

pred_df=pd.DataFrame(all_rows)
pred_df.to_csv(out_dir/"per_image_predictions.csv",index=False)

metrics=[]
for condition,g in pred_df.groupby("condition",sort=False):
    metrics.append({
        "condition":condition,"n":len(g),
        "accuracy":accuracy_score(g.label,g.prediction),
        "macro_f1":f1_score(g.label,g.prediction,average="macro",zero_division=0),
        "balanced_accuracy":balanced_accuracy_score(g.label,g.prediction),
        "mean_confidence":g.confidence.mean()
    })
metrics_df=pd.DataFrame(metrics)
metrics_df.to_csv(out_dir/"condition_metrics.csv",index=False)

# Paired prediction transitions relative to clean condition, by source ID.
wide=pred_df.pivot(index="source_id",columns="condition",values="correct")
transitions=[]
for condition in conditions:
    if condition=="clean": continue
    clean=wide["clean"].astype(bool); other=wide[condition].astype(bool)
    transitions.append({
        "condition_vs_clean":condition,
        "correct_to_wrong":int((clean & ~other).sum()),
        "wrong_to_correct":int((~clean & other).sum()),
        "both_correct":int((clean & other).sum()),
        "both_wrong":int((~clean & ~other).sum()),
        "prediction_changed":int((pred_df[pred_df.condition=="clean"].set_index("source_id").prediction !=
                                  pred_df[pred_df.condition==condition].set_index("source_id").prediction).sum())
    })
pd.DataFrame(transitions).to_csv(out_dir/"paired_transitions_vs_clean.csv",index=False)

# Per-condition, per-class metrics and confusion matrices.
class_rows=[]
for condition,g in pred_df.groupby("condition",sort=False):
    for ci,cls in enumerate(classes):
        z=g[g.label==ci]
        class_rows.append({"condition":condition,"class":cls,"n":len(z),
                           "accuracy":float((z.label==z.prediction).mean()) if len(z) else np.nan})
pd.DataFrame(class_rows).to_csv(out_dir/"per_class_accuracy.csv",index=False)

# Graphs
plt.figure(figsize=(12,5))
plt.bar(metrics_df.condition,metrics_df.accuracy)
plt.ylabel("Top-1 accuracy"); plt.xlabel("Condition"); plt.title("ResNet-18 accuracy under matched compound corruptions")
plt.xticks(rotation=35,ha="right"); plt.tight_layout()
plt.savefig(out_dir/"accuracy_by_condition.png",dpi=180); plt.show()

plt.figure(figsize=(12,5))
plt.bar(metrics_df.condition,metrics_df.macro_f1)
plt.ylabel("Macro-F1"); plt.xlabel("Condition"); plt.title("Macro-F1 under matched compound corruptions")
plt.xticks(rotation=35,ha="right"); plt.tight_layout()
plt.savefig(out_dir/"macro_f1_by_condition.png",dpi=180); plt.show()

# Save run metadata and summary.
metadata={
    "dataset_root":str(data_root),"checkpoint_path":str(ckpt_path),
    "n_images":len(records),"classes":classes,"conditions":conditions,
    "seed":SEED,"batch_size":BATCH_SIZE,"device":str(device),
    "preprocessing":"Resize(256), CenterCrop(224), ImageNet normalization",
    "corruptions":{"JPEG quality":50,"Gaussian noise sigma":12.0,
                   "Gaussian blur radius":1.2,"brightness multiplier":1.25},
    "note":"Evaluation only; no model training. Use test split only. These pilot corruptions are not a standardized benchmark."
}
with open(out_dir/"run_metadata.json","w") as f: json.dump(metadata,f,indent=2)
with open(out_dir/"summary.txt","w") as f:
    f.write("Matched compound-corruption pilot\n")
    f.write(metrics_df.to_string(index=False))
    f.write("\n\n"+pd.DataFrame(transitions).to_string(index=False))
elapsed=time.time()-start_time
print(f"\nFinished in {elapsed/60:.1f} minutes.")
print(metrics_df.to_string(index=False))
print("\nResults folder:",out_dir)

zip_path="/content/compound_robustness_results.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for fp in out_dir.rglob("*"):
        if fp.is_file(): z.write(fp,fp.relative_to(out_dir.parent))
print("ZIP:",zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Download manually from:",zip_path, "|",e)
